# Модуль 0–1. Облачная видеокарта и первый запуск нейросети YOLO

<a href="https://colab.research.google.com/github/temir8173/magistr/blob/main/practice/notebooks/module-01-intro-yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ноутбук работает в трёх местах: **Kaggle** (основная площадка), **Google Colab**
(запасная) и **на своём компьютере**. Как открыть его на Kaggle — в `README.md`.

**Зачем этот модуль.** До обучения своей модели нужно понять три вещи:
где и на чём она будет работать (облачная видеокарта), что такое изображение
для компьютера и что умеет готовая нейросеть. В конце модуля вы своими
глазами увидите главную проблему, которую решает диссертация.

**Время:** около недели по 1–2 часа.

**Что сдать** (подробно — в конце ноутбука):
1. этот ноутбук с результатами запуска;
2. ответы на контрольные вопросы — `practice/answers/module-01.md`;
3. запись в журнале — `practice/journal.md`;
4. картинки и таблицы — `practice/results/module-1/`.

## Как работать с ноутбуком

- Ноутбук состоит из **текстовых ячеек** (как эта) и **ячеек с кодом**.
- Ячейка с кодом запускается кнопкой ▶ слева или **Shift+Enter**.
  Результат появляется под ячейкой.
- Запускайте ячейки **по порядку сверху вниз**. Если что-то сломалось или
  сессия перезапустилась — запустите всё сверху заново
  (Kaggle: *Run All*; Colab: *Runtime → Run all*).
- Встречается значок **🔮 Предскажите** — *до* запуска ячейки подумайте и
  запишите в журнал, что ожидаете увидеть. Потом сравните. Это самая полезная
  часть обучения: так видно, где интуиция ошибается.
- Встречается значок **✍️ Задание** — нужно что-то изменить в коде самому.
- Встречается значок **❓ Вопрос** — ответ пишется в `practice/answers/module-01.md`.
- Ошибку в коде не бойтесь: прочитайте **последнюю строку** сообщения об ошибке —
  обычно там написано, что не так. Если непонятно — запишите в журнал и спросите.

---
# Часть 0. Где запускаем и зачем видеокарта

**Облачный ноутбук** — это компьютер в дата-центре, которым можно пользоваться
через браузер. Мы используем два таких сервиса:

- **Kaggle** (kaggle.com, принадлежит Google) — основная площадка: бесплатно
  около 30 часов видеокарты в неделю, сессия до 12 часов, обучение может идти
  в фоне при закрытом браузере.
- **Google Colab** — запасная: бесплатная видеокарта выдаётся не всегда.

Когда вы открываете ноутбук, сервис выделяет вам виртуальную машину
(«сессию»). Она **временная**: после закрытия все файлы, скачанные или
созданные в ней, пропадают. Поэтому результаты в конце модуля мы скачиваем.

**Зачем видеокарта (GPU).** Нейросеть — это миллионы простых умножений и
сложений. Процессор (CPU) делает их по очереди на нескольких ядрах.
Видеокарта имеет тысячи маленьких ядер и делает тысячи таких операций
одновременно. Для обучения нейросетей это ускорение в десятки раз.

### Включите видеокарту
- **Kaggle:** в правой панели *Session options* → **Accelerator: GPU T4 x2**
  (или *GPU P100*). Там же включите **Internet: On** — без него не скачаются
  библиотеки. Если этих пунктов нет — подтвердите номер телефона в настройках
  аккаунта Kaggle (см. README).
- **Colab:** меню *Runtime → Change runtime type → T4 GPU → Save*.
- **Свой компьютер:** видеокарта будет, только если она NVIDIA и установлен
  PyTorch с поддержкой CUDA. Без неё весь модуль тоже работает, просто медленнее.

Сначала узнаем, где именно запущен ноутбук.

In [ ]:
import os
import sys
from pathlib import Path

# Определяем площадку по её характерным признакам
try:
    import google.colab  # этот модуль есть только в Colab
    PLATFORM = "colab"
except ImportError:
    PLATFORM = "kaggle" if Path("/kaggle/input").exists() else "local"

print("Ноутбук запущен на площадке:", PLATFORM)

In [ ]:
# Восклицательный знак в начале — это команда не Python, а системы (Linux).
# nvidia-smi показывает, какая видеокарта подключена.
!nvidia-smi

**Как читать результат.** Ищите название карты (`Tesla T4` или `Tesla P100`)
и объём памяти (около `15–16 ГБ`). На Kaggle с *T4 x2* в таблице будут две карты.
Если вместо таблицы ошибка `command not found` или `NVIDIA-SMI has failed` —
видеокарта не включена, вернитесь к шагу выше.

In [ ]:
import sys
import torch  # PyTorch — библиотека, на которой работают нейросети YOLO

print("Версия Python:", sys.version.split()[0])
print("Версия PyTorch:", torch.__version__)
print("Видеокарта доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Название видеокарты:", torch.cuda.get_device_name(0))

### Эксперимент: CPU против GPU

Перемножим две большие таблицы чисел (матрицы 4096 × 4096). Это ровно та
операция, из которой в основном и состоит работа нейросети.

🔮 **Предскажите:** во сколько раз видеокарта окажется быстрее процессора?
В 2 раза? В 10? В 100? Запишите своё предположение в журнал.

In [ ]:
import time

def matmul_time(device, size=4096, repeats=5):
    """Среднее время (в секундах) перемножения двух матриц size x size."""
    a = torch.rand(size, size, device=device)
    b = torch.rand(size, size, device=device)
    a @ b  # «прогрев»: первый запуск всегда медленнее, его не считаем
    if device == "cuda":
        torch.cuda.synchronize()  # дождаться, пока видеокарта закончит
    start = time.perf_counter()
    for _ in range(repeats):
        a @ b
    if device == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - start) / repeats

cpu_t = matmul_time("cpu")
print(f"CPU: {cpu_t * 1000:.1f} мс на одно умножение")

if torch.cuda.is_available():
    gpu_t = matmul_time("cuda")
    print(f"GPU: {gpu_t * 1000:.1f} мс на одно умножение")
    print(f"Видеокарта быстрее в {cpu_t / gpu_t:.0f} раз")
else:
    print("GPU не включена — замер только для процессора.")

Запишите полученные числа в журнал. Ваше «во сколько раз» — это
аргумент, который можно привести на защите, объясняя, почему модели
обучались в Colab, а не на ноутбуке.

❓ **Вопрос 0.1.** Почему нейросети обучают на видеокарте, а не на процессоре?
Объясните своими словами и приведите свой замер.

---
# Часть 1. Изображение — это таблица чисел

Для компьютера цветная картинка — это трёхмерный массив чисел размером
**высота × ширина × 3**. Тройка — это три канала цвета: красный (R),
зелёный (G), синий (B). Каждое число — яркость канала от 0 (нет цвета)
до 255 (максимум).

Скачаем тестовую картинку и посмотрим на неё «глазами компьютера».

In [ ]:
import io
import numpy as np
import requests
from PIL import Image
import matplotlib.pyplot as plt

def load_image_from_url(url):
    """Скачивает картинку по ссылке и возвращает её как объект PIL.Image."""
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return Image.open(io.BytesIO(response.content)).convert("RGB")

# Тестовая картинка из документации YOLO (автобус и люди)
test_img = load_image_from_url("https://ultralytics.com/images/bus.jpg")
arr = np.array(test_img)  # превращаем картинку в массив чисел

print("Размер массива (высота, ширина, каналы):", arr.shape)
print("Тип чисел:", arr.dtype)
print("Минимум и максимум:", arr.min(), arr.max())
print("Пиксель в левом верхнем углу [R, G, B]:", arr[0, 0])

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
axes[0].imshow(arr)
axes[0].set_title("Исходная картинка")
for i, name in enumerate(["Красный (R)", "Зелёный (G)", "Синий (B)"]):
    axes[i + 1].imshow(arr[:, :, i], cmap="gray")  # один канал — серая картинка
    axes[i + 1].set_title(name)
for ax in axes:
    ax.axis("off")
plt.show()

Каждый канал — это отдельная «серая» картинка: светлее там, где этого цвета
больше. Сравните, например, жёлтые части автобуса в трёх каналах.

**Масштаб.** Типичный снимок с дрона — 5472 × 3648 пикселей.

❓ **Вопрос 1.5.** Сколько чисел хранит один такой цветной снимок?
Если на пастбище сделано 300 снимков — сколько это чисел? (Простая арифметика,
но она объясняет, почему без автоматизации тут не обойтись.)

### Рамка (bounding box) — это просто четыре числа

Чтобы указать, *где* на картинке объект, используют прямоугольник —
**рамку**. Её задают координатами левого верхнего угла `(x1, y1)` и правого
нижнего `(x2, y2)` в пикселях. Ось `x` идёт вправо, ось `y` — **вниз**
(начало координат в левом верхнем углу картинки).

Вырезать часть картинки по рамке — это просто взять кусок массива:
`arr[y1:y2, x1:x2]`.

In [ ]:
import matplotlib.patches as patches

# ✍️ ЗАДАНИЕ: подберите координаты так, чтобы рамка точно охватила
# человека слева на картинке. Меняйте числа и перезапускайте ячейку.
x1, y1, x2, y2 = 300, 50, 500, 250

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(arr)
axes[0].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                    linewidth=3, edgecolor="red", facecolor="none"))
axes[0].set_title("Картинка с рамкой")
axes[1].imshow(arr[y1:y2, x1:x2])  # вырезаем то, что внутри рамки
axes[1].set_title("Содержимое рамки")
plt.show()
print("Ширина рамки:", x2 - x1, "px, высота:", y2 - y1, "px")

Именно такие рамки нейросеть будет рисовать вокруг каждой коровы и овцы.
А в датасетах (модуль 2) люди заранее нарисовали такие рамки руками —
на них модель и учится.

---
# Часть 2. Классификация, детекция, сегментация

В компьютерном зрении три основные задачи:

| Задача | Что выдаёт | Пример ответа |
|---|---|---|
| **Классификация** | один ответ на всю картинку | «на фото — автобус» |
| **Детекция** | рамку + класс + уверенность для **каждого** объекта | «автобус здесь, человек здесь, человек здесь…» |
| **Сегментация** | точный контур (маску) каждого объекта, до пикселя | «вот эти пиксели — человек №1» |

Сейчас запустим все три на одной картинке. Для этого поставим библиотеку
**Ultralytics** — в ней собраны модели YOLO. Установка занимает около минуты.

In [ ]:
# %pip ставит библиотеку именно в то окружение, где работает ноутбук.
# На своём компьютере ultralytics уже стоит из requirements.txt — ячейка ничего не сломает.
%pip install -q ultralytics

In [ ]:
from ultralytics import YOLO

# Буква n = nano, самая маленькая и быстрая версия модели.
# При первом запуске файлы моделей (.pt) скачиваются автоматически.
model_cls = YOLO("yolo11n-cls.pt")   # классификация
model_det = YOLO("yolo11n.pt")       # детекция
model_seg = YOLO("yolo11n-seg.pt")   # сегментация

res_cls = model_cls(test_img, verbose=False)[0]
res_det = model_det(test_img, verbose=False)[0]
res_seg = model_seg(test_img, verbose=False)[0]

# Классификация: 5 самых вероятных ответов
print("Классификация, топ-5 ответов:")
for idx, p in zip(res_cls.probs.top5, res_cls.probs.top5conf.tolist()):
    print(f"  {res_cls.names[idx]:<20} {p:.2f}")

# .plot() рисует результат на картинке (возвращает её в формате BGR,
# поэтому [:, :, ::-1] переставляет каналы обратно в RGB для показа)
fig, axes = plt.subplots(1, 2, figsize=(14, 8))
axes[0].imshow(res_det.plot()[:, :, ::-1])
axes[0].set_title("Детекция")
axes[1].imshow(res_seg.plot()[:, :, ::-1])
axes[1].set_title("Сегментация")
for ax in axes:
    ax.axis("off")
plt.show()

Обратите внимание: классификация ответила одним словом на всю картинку и
**ничего не сказала о том, сколько на ней людей**. Детекция нарисовала рамку
вокруг каждого. Сегментация — ещё и точный контур.

Для подсчёта поголовья нужна именно **детекция**: каждая рамка — одно животное,
число рамок — число голов. Сегментация дала бы больше информации, но для её
обучения нужна разметка контуров каждого животного — это в разы дороже,
а для подсчёта избыточно.

❓ **Вопрос 1.1.** Чем детекция отличается от классификации и сегментации?
Почему для подсчёта скота выбрана детекция?

---
# Часть 3. Нейросеть «на пальцах»

**Нейросеть** — это очень большая формула с миллионами настраиваемых чисел
(**весов**). На вход — числа картинки, на выход — рамки и классы.
Сначала веса случайные, и формула выдаёт мусор.

**Обучение** — это повторение одного цикла много тысяч раз:
1. показать сети картинку, для которой правильный ответ известен (разметка);
2. посмотреть, насколько ответ сети отличается от правильного (**ошибка**, *loss*);
3. чуть-чуть подвинуть все веса в ту сторону, где ошибка меньше;
4. взять следующую картинку.

Один проход по всем картинкам датасета называется **эпохой**.
Обучение обычно длится десятки эпох.

Модель `yolo11n.pt`, которую мы запускали, **уже обучена** разработчиками на
датасете **COCO**: около 118 тысяч фотографий с размеченными объектами
80 классов. Посмотрим, какие классы она знает.

In [ ]:
print(f"Модель знает {len(model_det.names)} классов:")
print(model_det.names)

In [ ]:
# ✍️ ЗАДАНИЕ: найдите в списке выше номера классов для коровы, овцы и лошади
# и впишите их сюда вместо None.
COW_ID = None
SHEEP_ID = None
HORSE_ID = None

for name, idx in [("cow", COW_ID), ("sheep", SHEEP_ID), ("horse", HORSE_ID)]:
    if idx is None:
        print(f"{name}: номер ещё не вписан")
    else:
        status = "верно" if model_det.names[idx] == name else "неверно, проверьте"
        print(f"{name}: {idx} — {status}")

Итак, готовая модель «знает» коров и овец. Казалось бы, обучать ничего не
надо — бери и считай. Проверим это в следующей части.

---
# Часть 4. Первый подсчёт животных

Нужны **три картинки**:

1. **Обычная фотография коров или овец, снятая с земли, сбоку** — найдите
   сами: Wikimedia Commons (commons.wikimedia.org), Unsplash, Pexels — по
   запросам *cows pasture*, *sheep flock*.
2. **`aerial_cattle.jpg`** — настоящий снимок с дрона: пастбище с коровами.
3. **`aerial_sheep.jpg`** — настоящий снимок с дрона: отара овец.

Снимки 2 и 3 взяты из тех датасетов, на которых мы будем обучать свою модель.
Они лежат в репозитории в папке `practice/data/samples/` (там же описание и
авторы). Для каждого известен **правильный ответ** — сколько животных на нём
разметили люди: **53 коровы** и **50 овец**.

Как дать ноутбуку эти файлы — зависит от площадки:

- **Kaggle:** в правой панели *Input* → **Upload** → *New Dataset* → выберите
  все три файла → название `magistr-samples` → *Create*. Файлы появятся в папке
  `/kaggle/input/`, код ниже найдёт их сам. Датасет сохранится в вашем аккаунте,
  в следующий раз его можно просто подключить через *Add Input*.
- **Colab:** ячейка ниже покажет кнопку «Выбрать файлы» — выберите все три.
- **Свой компьютер:** снимки с дрона уже лежат в `practice/data/samples/`.
  Своё фото положите в папку `practice/results/module-1/`.

Если какой-то файл не найдётся, код подставит тестовую картинку с автобусом,
чтобы всё работало дальше, но выводы этой части без настоящих снимков
сделать не получится.

In [ ]:
# ✍️ ЗАДАНИЕ: впишите имя файла своего фото с земли.
GROUND_PHOTO = "my_cows.jpg"
AERIAL_CATTLE = "aerial_cattle.jpg"
AERIAL_SHEEP = "aerial_sheep.jpg"

# Правильные ответы — число животных в разметке датасетов
TRUE_COUNTS = {AERIAL_CATTLE: 53, AERIAL_SHEEP: 50}

In [ ]:
def find_file(name):
    """Ищет файл в местах, где он может быть на разных площадках. Возвращает путь или None."""
    for folder in [Path("."), Path("../data/samples"), Path("../results/module-1"), Path("/content")]:
        if (folder / name).exists():
            return folder / name
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        found = list(kaggle_input.rglob(name))  # rglob ищет во всех вложенных папках
        if found:
            return found[0]
    return None

needed = [GROUND_PHOTO, AERIAL_CATTLE, AERIAL_SHEEP]
images = {}
for name in needed:
    path = find_file(name)
    if path:
        images[name] = Image.open(path).convert("RGB")
        print(f"✅ {name}: {path}")

missing = [n for n in needed if n not in images]
if missing and PLATFORM == "colab":
    from google.colab import files
    print("Не найдены:", missing, "— выберите их на компьютере:")
    for name, data in files.upload().items():
        images[name] = Image.open(io.BytesIO(data)).convert("RGB")

for name in needed:
    if name not in images:
        print(f"⚠️ Не найден {name} — вместо него тестовая картинка с автобусом.")
        images[name] = test_img

In [ ]:
from collections import Counter

def detect_and_count(img, conf=0.25, title=""):
    """Запускает детекцию, рисует результат и возвращает ({класс: количество}, результат)."""
    result = model_det(img, conf=conf, verbose=False)[0]
    class_ids = result.boxes.cls.int().tolist()        # номер класса каждой рамки
    counts = Counter(result.names[i] for i in class_ids)

    plt.figure(figsize=(12, 8))
    plt.imshow(result.plot()[:, :, ::-1])
    plt.title(f"{title}  |  порог={conf}  |  найдено: {dict(counts) or 'ничего'}")
    plt.axis("off")
    plt.show()
    return counts, result

### Фото с земли

🔮 **Предскажите:** найдёт ли модель всех животных? Сначала посчитайте их
**сами, глазами** и запишите в журнал — это правильный ответ для вашего фото.

In [ ]:
counts_ground, res_ground = detect_and_count(images[GROUND_PHOTO], title="Фото с земли")
print(counts_ground)

### Снимки с дрона

🔮 **Предскажите:** модель знает классы `cow` и `sheep` и, скорее всего,
неплохо справилась с фото с земли. Сколько коров из 53 и сколько овец из 50
она найдёт на снимках с дрона? Запишите предсказание в журнал.

In [ ]:
import pandas as pd  # pandas — библиотека для таблиц

aerial_results = {}
rows = []
for name in [AERIAL_CATTLE, AERIAL_SHEEP]:
    counts, res = detect_and_count(images[name], title=f"Снимок с дрона: {name}")
    aerial_results[name] = res
    rows.append({
        "снимок": name,
        "правильно (разметка)": TRUE_COUNTS[name],
        "найдено cow": counts.get("cow", 0),
        "найдено sheep": counts.get("sheep", 0),
        "найдено другого": sum(v for k, v in counts.items() if k not in ("cow", "sheep")),
        "что именно": dict(counts) or "ничего",
    })

aerial_table = pd.DataFrame(rows)
display(aerial_table)

Посмотрите на картинки и таблицу внимательно. Готовая модель, которая
«знает» коров и овец, на снимках с дрона почти не находит ни тех, ни других.
Коров она может вообще не увидеть, а овец — принять за что-то совсем другое
(посмотрите на подписи рамок).

**Почему так.** В датасете COCO почти все фотографии сняты людьми с земли.
Корова там — это силуэт сбоку: голова, ноги, рога. Сверху корова выглядит как
небольшое тёмное или пятнистое пятно (на снимке `aerial_cattle.jpg` — всего
несколько десятков пикселей), а овца — как светлый вытянутый овал. Таких
примеров модель никогда не видела, поэтому не узнаёт их.

**Это и есть главная проблема, которую решает практическая часть
диссертации:** готовые модели не подходят для снимков с дрона, их нужно
**дообучить** на аэроснимках скота (модули 2–4). Эти две картинки и таблицу
стоит сохранить — они пригодятся в главе как иллюстрация проблемы.

❓ **Вопрос 1.3.** Почему готовая модель плохо находит животных на снимках
с дрона, хотя знает классы `cow` и `sheep`? Что нужно сделать, чтобы это исправить?

---
# Часть 5. Порог уверенности (confidence)

Для каждой рамки модель выдаёт **уверенность** — число от 0 до 1:
насколько модель уверена, что в рамке действительно объект этого класса.
Рамки с уверенностью ниже **порога** (`conf`, по умолчанию 0.25)
отбрасываются и не показываются.

Посмотрим на уверенность каждой рамки на вашем фото с земли.

In [ ]:
for box_conf, cls_id in zip(res_ground.boxes.conf.tolist(), res_ground.boxes.cls.int().tolist()):
    print(f"{res_ground.names[cls_id]:<12} уверенность {box_conf:.2f}")

Теперь проведём опыт на снимке с овцами.

🔮 **Предскажите:** что произойдёт с числом рамок, если порог понизить
до 0.05? А если повысить до 0.75?

In [ ]:
# ✍️ ЗАДАНИЕ: после первого запуска повторите опыт на других снимках
# (GROUND_PHOTO, AERIAL_CATTLE) и сравните.
EXPERIMENT_PHOTO = AERIAL_SHEEP
thresholds = [0.05, 0.10, 0.25, 0.50, 0.75]

rows = []
for t in thresholds:
    r = model_det(images[EXPERIMENT_PHOTO], conf=t, verbose=False)[0]
    counts = Counter(r.names[i] for i in r.boxes.cls.int().tolist())
    rows.append({"порог": t, "всего рамок": len(r.boxes), **counts})

table = pd.DataFrame(rows).fillna(0)
display(table)

plt.figure(figsize=(7, 4))
plt.plot(table["порог"], table["всего рамок"], marker="o")
if EXPERIMENT_PHOTO in TRUE_COUNTS:
    plt.axhline(TRUE_COUNTS[EXPERIMENT_PHOTO], color="green", linestyle="--",
                label="правильный ответ (разметка)")
    plt.legend()
plt.xlabel("Порог уверенности (conf)")
plt.ylabel("Найдено рамок")
plt.title(f"Зависимость числа рамок от порога: {EXPERIMENT_PHOTO}")
plt.grid(True)
plt.show()

In [ ]:
# Посмотрите глазами на крайние случаи
detect_and_count(images[EXPERIMENT_PHOTO], conf=0.05, title="Низкий порог")
detect_and_count(images[EXPERIMENT_PHOTO], conf=0.75, title="Высокий порог");

При низком пороге появляется больше рамок — в том числе **лишние**: на траве,
дублирующие, с неверным классом. При высоком часть объектов **пропадает**.
Подсчёт поголовья напрямую зависит от порога. В модуле 4 это превратится
в две важнейшие метрики — **Precision** (какая доля найденного верна)
и **Recall** (какая доля существующего найдена).

Обратите внимание на снимок с овцами при низком пороге: рамок становится
примерно столько же, сколько овец на самом деле, но класс у них неверный.

❓ **Вопрос 1.2.** Что означает уверенность 0.45 у рамки?

❓ **Вопрос 1.4.** Как менялось число найденных объектов при изменении
порога и почему? Можно ли «починить» подсчёт с дрона, просто подобрав порог?

❓ **Вопрос 1.6.** При низком пороге на снимке с овцами модель находит
примерно столько объектов, сколько там овец, но называет их неправильно.
О чём это говорит? Почему это хорошая новость для дообучения?

---
# Часть 6. Сохранение результатов

Эти картинки и таблицы пригодятся в главе диссертации (раздел 3.1 —
иллюстрация проблемы). Сохраним их.

In [ ]:
import shutil

# На своём компьютере сохраняем прямо в папку репозитория,
# в облаке — в рабочую папку сессии, а потом скачиваем архивом.
if PLATFORM == "local":
    out_dir = Path("../results/module-1")
elif PLATFORM == "kaggle":
    out_dir = Path("/kaggle/working/results/module-1")
else:
    out_dir = Path("/content/results/module-1")
out_dir.mkdir(parents=True, exist_ok=True)

Image.fromarray(res_ground.plot()[:, :, ::-1]).save(out_dir / "ground_detection.jpg", quality=85)
for name, res in aerial_results.items():
    Image.fromarray(res.plot()[:, :, ::-1]).save(out_dir / f"pretrained_{name}", quality=85)
aerial_table.to_csv(out_dir / "pretrained_aerial_counts.csv", index=False)
table.to_csv(out_dir / "confidence_threshold.csv", index=False)
print("Сохранено в", out_dir, ":", sorted(p.name for p in out_dir.iterdir()))

if PLATFORM != "local":
    archive = shutil.make_archive(str(out_dir.parent.parent / "module-1-results"), "zip", out_dir)
    print("Архив:", archive)
    if PLATFORM == "colab":
        from google.colab import files
        files.download(archive)

- **Свой компьютер:** всё уже лежит в `practice/results/module-1/`.
- **Kaggle:** в правой панели *Output* (папка `/kaggle/working`) скачайте
  `module-1-results.zip` и распакуйте в `practice/results/module-1/`.
- **Colab:** архив скачается сам — распакуйте его туда же.

---
# Итог модуля и что сдать

**Что вы теперь знаете:**
- что такое облачный ноутбук, видеокарта и почему нейросети обучают на GPU;
- что картинка — это массив чисел, а рамка — четыре координаты;
- чем отличаются классификация, детекция и сегментация;
- как в общих чертах обучается нейросеть;
- что готовая YOLO знает коров и овец, но **не узнаёт их на настоящих снимках
  с дрона** — и поэтому её нужно дообучать;
- как порог уверенности влияет на подсчёт.

### Сдать
1. **Ноутбук с результатами** — заменить им файл
   `practice/notebooks/module-01-intro-yolo.ipynb` в своей ветке `module-1`:
   - Kaggle: *File → Download notebook*;
   - Colab: *File → Download → .ipynb*;
   - свой компьютер: просто сохранить.
2. **Ответы:** скопировать `practice/answers/_template.md` в
   `practice/answers/module-01.md` и ответить на вопросы:
   - **0.1** Почему нейросети обучают на видеокарте, а не на процессоре? Ваш замер.
   - **1.1** Чем детекция отличается от классификации и сегментации?
     Почему для подсчёта скота выбрана детекция?
   - **1.2** Что означает уверенность 0.45 у рамки?
   - **1.3** Почему готовая модель плохо находит животных на снимках с дрона,
     хотя знает классы `cow` и `sheep`? Что нужно сделать?
   - **1.4** Как менялось число найденных объектов при изменении порога и почему?
     Можно ли «починить» подсчёт с дрона, просто подобрав порог?
   - **1.5** Сколько чисел хранит снимок 5472 × 3648 RGB? А 300 таких снимков?
   - **1.6** При низком пороге на снимке с овцами модель находит примерно
     столько объектов, сколько там овец, но называет их неправильно.
     О чём это говорит? Почему это хорошая новость для дообучения?
3. **Журнал:** запись по шаблону в `practice/journal.md` — ваши предсказания (🔮),
   что получилось, что удивило, что непонятно.
4. **Результаты:** `practice/results/module-1/` — картинки и таблицы из части 6.
5. `commit` → `push` → **Pull Request** в `main` (команды — в README.md).

### Почитать / посмотреть (по желанию)
- Документация Ultralytics: https://docs.ultralytics.com — разделы *Quickstart* и *Predict*.
- 3Blue1Brown, «But what is a neural network?» (YouTube, есть русские субтитры) —
  лучшее наглядное объяснение того, как устроена нейросеть.